In [ ]:
%load_ext autoreload
%autoreload 2
import ezy_seq as ezy
import scanpy as sc
import pandas as pd
from pathlib import Path  
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.path import Path as MPath 
import random
random.seed(42)

In [ ]:
adata_full=sc.read_h5ad(r"/path/to/adata.h5ad")

In [ ]:
sc.pp.scale(
    adata_full,
    zero_center=True,       # Subtract the mean from each gene (result has 0 mean)
    max_value=10,           # Clip values exceeding this standard deviation (prevents outliers from dominating)
    copy=False,             # Modify the AnnData object in place
    layer=None,             # Scale a specific layer instead of .X
    obsm=None               # Scale a specific multi-dimensional observation annotation
)

In [ ]:
sub=adata_full[adata_full.obs['ct_simple'].isin(['Astrocytes','Cholinergic.neurons','Microglia','Inhibitory.neurons','Mature.oligodendrocytes','Excitatory.neurons'])]


In [ ]:

#sub=adata_full[adata_full.obs['ct_simple'].isin(['Cholinergic.neurons', 'Excitatory.neurons', 'Inhibitory.neurons', 'Mature.oligodendrocytes', 'Microglia', 'Astrocytes'])]
sub=adata_full[adata_full.obs['FMT'].isin(['Healthy_FMT','Stroke_FMT'])]
sub=sub[sub.obs['napari_regioin']!='Olfactory']


In [ ]:
import scanpy as sc


print('--- Step 1: PCA (Principal Component Analysis) ---')
# Parameters are based on sc.tl.pca (which uses sklearn or arpack)

print('--- Step 1: PCA ---')
sc.tl.pca(
    sub, 
    n_comps=15,                 
    zero_center=True, 
    svd_solver='arpack', 
    random_state=0, 
    use_highly_variable=False,   
    dtype='float32', 
    copy=False
)

print('--- Step 2: Neighbors ---')
sc.pp.neighbors(
    sub, 
    n_neighbors=10,              # Keep this low for tight clusters
    n_pcs=15,                   
    knn=True, 
    random_state=0, 
    method='umap', 
    metric='cosine',             
    copy=False 
)

print('--- Step 3: UMAP ---')
# This computes the embedding itself
sc.tl.umap(
    sub, 
    min_dist=0.05,                # Effective minimum distance between embedded points
    spread=1.5,                  # Effective scale of embedded points
    n_components=2,              # The dimension of the space to embed into
    maxiter=None,                # Max number of iterations for optimization (None = auto)
    alpha=1.0,                   # Initial learning rate for the optimization
    gamma=1.0,                   # Weighting of the negative samples
    negative_sample_rate=5,      # Number of negative samples to select per positive sample
    init_pos='spectral',         # How to initialize the low dimensional embedding
    random_state=0,              # Seed for the random number generator
    a=None,                      # Parameter 'a' of the UMAP curve (None = inferred)
    b=None,                      # Parameter 'b' of the UMAP curve (None = inferred)
    copy=False,                  # Return a copy or modify in place
    method='umap',               # UMAP implementation ('umap' or 'rapids')
    neighbors_key=None           # Key in adata.uns where neighbor info is stored
)

print('--- Visualization ---')
sc.pl.umap(sub,color='ct_simple')

In [ ]:
import re
import pandas as pd

_NORM = {
    "fiber tracts":                    "Fiber tract",
    "fiber tract - lateral ventricle": "Fiber tract",
    "fiber tract":                     "Fiber tract",
    "denate gyrus":                    "Dentate Gyrus",
    "medulla":                         "Medulla",
}

_PREFIX_MAP = {
    "midbrain": "Midbrain",
}

def clean_one(region):
    if pd.isna(region):
        return pd.NA
    r = str(region).strip()
    if re.fullmatch(r"[\d,\s]+", r):
        return pd.NA
    if "," in r:
        r = r.split(",")[0].strip()
    for prefix, canonical in _PREFIX_MAP.items():
        if r.lower().startswith(prefix):
            return canonical
    canonical = _NORM.get(r.lower())
    if canonical:
        return canonical
    return r

sub.obs['quint_region_clean'] = sub.obs['quint_region'].map(clean_one)
sub = sub[sub.obs['quint_region_clean'].notna()].copy()

In [ ]:
list(sub.obs['quint_region'].unique())

In [ ]:
sc.pl.umap(sub,color='ct_simple',title='ct_simple')
sc.pl.umap(sub,color='cell_type',title='cell_type')
sc.pl.umap(sub,color='sample_ID',title='sample_ID')
sc.pl.umap(sub,color='FMT',title='FMT')
sc.pl.umap(sub,color='napari_region',title='napari_region')

sc.pl.umap(sub,color='ct_simple',title='ct_simple',legend_loc=None)
sc.pl.umap(sub,color='cell_type',title='cell_type',legend_loc=None)
sc.pl.umap(sub,color='sample_ID',title='sample_ID',legend_loc=None)
sc.pl.umap(sub,color='FMT',title='FMT',legend_loc=None)
sc.pl.umap(sub,color='quint_region_clean',title='quint_region_clean',legend_loc=None)

In [ ]:
sc.pl.umap(sub,color='quint_region',title='quint_region',legend_loc=None)